<a href="https://colab.research.google.com/github/pedrorostagno/tesis/blob/main/Tesis_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Usar A100, es el que mejor resultados me dio hasta ahora

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

files = os.listdir('/content/drive/Othercomputers/My Mac/Data/train/train/spoof/part_001')
print(len(files))


1000


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import os
from PIL import Image
import numpy as np
import pickle
from datetime import datetime

dataset_path = '/content/drive/Othercomputers/My Mac/Data/train'

# 🗂️ Carpetas base en Drive
BASE_DIR = "/content/drive/MyDrive/Tesis UTDT"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
EXPERIMENTS_DIR = os.path.join(BASE_DIR, "experimentos")

# ⏱️ Nombre único del experimento basado en fecha y hora
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = os.path.join(EXPERIMENTS_DIR, timestamp)

# 📁 Crear carpetas si no existen
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

print(f"📁 Cachés en:     {CACHE_DIR}")
print(f"📁 Resultados en: {RUN_DIR}")


📁 Cachés en:     /content/drive/MyDrive/Tesis UTDT/cache
📁 Resultados en: /content/drive/MyDrive/Tesis UTDT/experimentos/20250917-205412


In [4]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
import random
import os
import cv2  # 👈 importante para leer imágenes y recortar

class CelebASpoofDatasetFromCSV(Dataset):
    def __init__(self, csv_path, transform=None, max_samples=None,
                 shuffle=True, seed=42, balanced=False, use_bb=False):
        """
        Dataset desde CSV con opción de balanceo y recorte por bounding box.

        CSV esperado con columnas: 'path', 'label'
        - label: 0 (spoof), 1 (live)

        Parámetros extra:
        - use_bb: si True, recorta la cara usando los bounding boxes (_BB.txt)
        """
        self.transform = transform
        self.use_bb = use_bb
        df = pd.read_csv(csv_path)
        random.seed(seed)

        if balanced:
            spoof_df = df[df['label'] == 0]
            live_df  = df[df['label'] == 1]
            min_len = min(len(spoof_df), len(live_df))

            spoof_df = spoof_df.sample(n=min_len, random_state=seed)
            live_df  = live_df.sample(n=min_len, random_state=seed)
            df = pd.concat([spoof_df, live_df], ignore_index=True)

            print(f"⚖️ Dataset balanceado: {min_len} spoof + {min_len} live = {2 * min_len} total")

        if shuffle:
            df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

        if max_samples:
            df = df.head(max_samples)

        self.image_paths = df['path'].tolist()
        self.labels = df['label'].tolist()

        print(f"📄 CSV cargado: {csv_path}")
        print(f"📊 Total imágenes cargadas: {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        filename = self.image_paths[idx]
        label = self.labels[idx]

        if self.use_bb:
            # Leer imagen con OpenCV
            img = cv2.imread(filename)
            real_h, real_w, c = img.shape

            # Archivo con bounding box
            bb_file = filename[:-4] + "_BB.txt"
            assert os.path.exists(bb_file), f"Bounding box file not found: {bb_file}"

            with open(bb_file, "r") as f:
                x, y, w, h, score = f.readline().strip().split(" ")

            # Convertir a int + escalar a resolución original
            x, y, w, h = map(float, (x, y, w, h))
            x = int(x * real_w / 224)
            y = int(y * real_h / 224)
            w = int(w * real_w / 224)
            h = int(h * real_h / 224)

            # Asegurar límites válidos
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(real_w, x1 + w), min(real_h, y1 + h)

            img = img[y1:y2, x1:x2, :]
            assert img.shape[0] > 0 and img.shape[1] > 0, f"Invalid crop: {filename}"

            # Convertir a PIL
            img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        else:
            # Cargar imagen normal con PIL
            img = Image.open(filename).convert("RGB")

        # Aplicar transformaciones (ImageNet en tu caso)
        if self.transform:
            img = self.transform(img)

        return img, label


In [5]:

from torchvision.transforms import InterpolationMode as IM

# 📁 Definición de rutas a cada partición del dataset (entrenamiento, validación y test)
train_data_dir = '/content/drive/Othercomputers/My Mac/Data/train/train'
val_data_dir   = '/content/drive/Othercomputers/My Mac/Data/train/val'
test_data_dir  = '/content/drive/Othercomputers/My Mac/Data/train/test'

# 📐 Dimensiones estándar a las que se redimensionarán todas las imágenes
img_width, img_height = 224, 224

# ⚙️ Tamaño del batch para los DataLoaders
batch_size = 64

# 🌀 Transformaciones de preprocesamiento para cada conjunto de datos
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((img_width, img_height), interpolation=IM.BILINEAR),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],             # Normalización con media y desvío estándar de ImageNet
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((img_width, img_height)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((img_width, img_height)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}


In [6]:
# Cantidad máxima de muestras por partición
train_max_samples = 10000
val_max_samples   = 5000
test_max_samples  = 10000

# 📦 Dataset de entrenamiento desde CSV
train_dataset = CelebASpoofDatasetFromCSV(
    csv_path='/content/drive/Othercomputers/My Mac/Data/train/train_image_list_balanced.csv',
    transform=data_transforms['train'],
    max_samples=train_max_samples,
    balanced=True,
    use_bb=True
)

# 📦 Dataset de validación desde CSV
val_dataset = CelebASpoofDatasetFromCSV(
    csv_path='/content/drive/Othercomputers/My Mac/Data/train/val_image_list.csv',
    transform=data_transforms['val'],
    max_samples=val_max_samples,
    balanced=True,
    use_bb=True
)

# 🔄 DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    drop_last=True  # Descarto el ultimo batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)



⚖️ Dataset balanceado: 25000 spoof + 25000 live = 50000 total
📄 CSV cargado: /content/drive/Othercomputers/My Mac/Data/train/train_image_list_balanced.csv
📊 Total imágenes cargadas: 10000
⚖️ Dataset balanceado: 23863 spoof + 23863 live = 47726 total
📄 CSV cargado: /content/drive/Othercomputers/My Mac/Data/train/val_image_list.csv
📊 Total imágenes cargadas: 5000


In [7]:
labels = np.array(train_dataset.labels)
unique, counts = np.unique(labels, return_counts=True)
print(f"Distribución de clases en entrenamiento: {dict(zip(unique, counts))}")

Distribución de clases en entrenamiento: {np.int64(0): np.int64(5022), np.int64(1): np.int64(4978)}


In [16]:
# ─── 1) MODELO  ───────────────────────────────────────────────
from torchvision import models
from torchvision.models import MobileNet_V2_Weights

model = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 2)       # 2 clases

print(model.classifier[1])
# (opcional) descongela todo el backbone para fine-tuning completo
for p in model.parameters():
    p.requires_grad = False # True

for p in model.classifier.parameters():
    p.requires_grad = True

Linear(in_features=1280, out_features=2, bias=True)


In [17]:
n_total  = sum(1 for _ in model.parameters())
n_train  = sum(1 for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales: {n_total} — entrenables: {n_train}")


Parámetros totales: 158 — entrenables: 2


In [18]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.AdamW(trainable_params, lr=5e-4, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)


In [12]:
# ─── Setup común ──────────────────────────────────────────────
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torch import amp
from tqdm import tqdm                # ⬅ progreso nice
import os, json, sklearn.metrics as skm
from collections import OrderedDict

use_cuda  = (device.type == "cuda")
amp_dtype = torch.float16            # o torch.bfloat16 (A100 lo soporta)

# ──────────────────────────────────────────────────────────────
# Helpers genéricos
# ──────────────────────────────────────────────────────────────
def _run_one_epoch(model, loader, criterion, device, scaler=None,
                   optimizer=None, train=False, max_grad_norm=None,
                   accum_steps=1):
    """
    Ejecuta una época completa.
    * Si `train=True` espera un optimizer y hace backward().
    * Si scaler es None → FP32 puro.
    """
    if train:
        model.train()
    else:
        model.eval()

    loss_sum, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []

    loop = tqdm(loader, leave=False, desc="train" if train else "val")
    optimizer_steps = 0

    for step, (x, y) in enumerate(loop, 1):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with amp.autocast(enabled=use_cuda, device_type="cuda", dtype=amp_dtype):
            logits = model(x)
            loss   = criterion(logits, y) / accum_steps

        if train:
            if scaler and scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            # Grad accumulation
            if step % accum_steps == 0:
                if max_grad_norm:
                    if scaler and scaler.is_enabled():
                        scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

                if scaler and scaler.is_enabled():
                    scaler.step(optimizer); scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_steps += 1

        loss_sum += loss.item() * x.size(0) * accum_steps
        preds     = logits.argmax(1)
        correct  += (preds == y).sum().item()
        total    += x.size(0)

        if not train:
            y_true.append(y.cpu())
            y_pred.append(preds.cpu())

    acc = correct / max(1, total)
    avg_loss = loss_sum / max(1, total)

    if train:
        return avg_loss, acc
    else:
        y_true = torch.cat(y_true).numpy()
        y_pred = torch.cat(y_pred).numpy()
        report = skm.classification_report(
            y_true, y_pred, output_dict=True, zero_division=0
        )
        return avg_loss, acc, report

# ──────────────────────────────────────────────────────────────
# Entrenamiento principal
# ──────────────────────────────────────────────────────────────
def train_model(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler, device,
        num_epochs=20, patience=6, checkpoint_interval=1,
        log_dir=None, max_grad_norm=None, accum_steps=1):

    # TensorBoard
    log_dir = log_dir or os.path.join(RUN_DIR, "tensorboard")
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)

    # Guardamos config reproducible
    config = OrderedDict(
        num_epochs=num_epochs,
        patience=patience,
        batch_size=train_loader.batch_size,
        optimizer=type(optimizer).__name__,
        lr_start=optimizer.param_groups[0]["lr"],
        grad_clip=max_grad_norm,
        accum_steps=accum_steps,
        torch_version=torch.__version__
    )
    json.dump(config, open(os.path.join(RUN_DIR, "config.json"), "w"), indent=2)

    scaler        = amp.GradScaler(enabled=use_cuda)
    best_val_loss = float("inf")
    wait          = 0

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs} — LR {scheduler.get_last_lr()[0]:.6f}")

        tr_loss, tr_acc = _run_one_epoch(
            model, train_loader, criterion, device,
            scaler=scaler, optimizer=optimizer, train=True,
            max_grad_norm=max_grad_norm, accum_steps=accum_steps
        )
        val_loss, val_acc, report = _run_one_epoch(
            model, val_loader, criterion, device, train=False
        )

        scheduler.step()

        print(f"TrainLoss {tr_loss:.4f}  ValLoss {val_loss:.4f}  "
              f"ValAcc {val_acc:.4f}  F1_spoof {report['0']['f1-score']:.3f}")

        # TensorBoard
        writer.add_scalars("Loss", {"train": tr_loss, "val": val_loss}, epoch)
        writer.add_scalars("Acc",  {"train": tr_acc , "val": val_acc }, epoch)

        # Early stopping + best checkpoint
        if val_loss < best_val_loss:
            best_val_loss, wait = val_loss, 0
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "scaler_state": scaler.state_dict(),
                "val_loss": val_loss
            }, os.path.join(RUN_DIR, "best_model.pth"))
            print("✅  Nuevo mejor modelo guardado")
        else:
            wait += 1
            print(f"⏳  Sin mejora ({wait}/{patience})")
            if wait >= patience:
                print("🛑  Early stopping")
                break

        # Checkpoint periódico
        if epoch % checkpoint_interval == 0:
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "scaler_state": scaler.state_dict(),
                "val_loss": val_loss
            }, os.path.join(RUN_DIR, f"ckpt_{epoch:02d}.pth"))

    writer.close()
    return model


In [ ]:
import os, json, torch, sklearn.metrics as metrics
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torch import amp                      # 👈  nueva forma
from collections import OrderedDict



# ──────────────────────────────────────────────────────────────
# 2) BUCLE DE ENTRENAMIENTO (precisión mixta moderna)
# ──────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with amp.autocast(device_type='cuda'):
            logits = model(x)
            loss   = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        total    += x.size(0)

    return loss_sum / total, correct / total


def validate_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with amp.autocast(device_type='cuda'):
                logits = model(x)
                loss   = criterion(logits, y)

            loss_sum += loss.item() * x.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            total    += x.size(0)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(logits.argmax(1).cpu().numpy())

    report = metrics.classification_report(
        y_true, y_pred, output_dict=True, zero_division=0
    )
    return loss_sum / total, correct / total, report

# ──────────────────────────────────────────────────────────────
# 3) FUNCIÓN PRINCIPAL DE ENTRENAMIENTO
# ──────────────────────────────────────────────────────────────
def train_model(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler, device,
        num_epochs=20, patience=6, checkpoint_interval=1, log_dir=None):

    if log_dir is None:
        log_dir = os.path.join(RUN_DIR, "tensorboard")
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)

    # guardar config
    json.dump(OrderedDict(
        num_epochs=num_epochs,
        patience=patience,
        batch_size=train_loader.batch_size,
        optimizer=type(optimizer).__name__,
        lr_start=optimizer.param_groups[0]["lr"]
    ), open(os.path.join(RUN_DIR, "config.json"), "w"), indent=2)

    scaler          = amp.GradScaler()
    best_val_loss   = float('inf')
    wait            = 0

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}\n" + "-" * 20)

        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer,
                                      criterion, device, scaler)
        val_loss, val_acc, report = validate_epoch(
            model, val_loader, criterion, device)

        scheduler.step()   # CosineAnnealingLR no necesita métricas

        print(f"LR: {scheduler.get_last_lr()[0]:.6f}  "
              f"TrainLoss: {tr_loss:.4f}  ValLoss: {val_loss:.4f}")
        print(f"Val Acc: {val_acc:.4f}  F1_spoof: {report['0']['f1-score']:.3f}")

        # logs tensorboard
        writer.add_scalars("Loss", {"train": tr_loss, "val": val_loss}, epoch)
        writer.add_scalars("Acc",  {"train": tr_acc , "val": val_acc }, epoch)

        # early-stopping
        if val_loss < best_val_loss:
            best_val_loss, wait = val_loss, 0
            torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_model.pth"))
            print("✅  Nuevo mejor modelo guardado")
        else:
            wait += 1
            print(f"⏳  Sin mejora ({wait}/{patience})")
            if wait >= patience:
                print("🛑  Early stopping")
                break

        # checkpoint periódico
        if epoch % checkpoint_interval == 0:
            torch.save({"epoch": epoch,
                        "model": model.state_dict(),
                        "optim": optimizer.state_dict()},
                       os.path.join(RUN_DIR, f"ckpt_{epoch:02d}.pth"))
    writer.close()
    return model


In [ ]:
# ──────────────────────────────────────────────────────────────
# FUNCIÓN DE PÉRDIDA – FOCAL LOSS
# ──────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=(0.25, 0.75), gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = torch.tensor(alpha)
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        # mueve alpha al device del tensor de logits
        if self.alpha.device != logits.device:
            self.alpha = self.alpha.to(logits.device)

        ce_loss = self.ce(logits, targets)
        pt      = torch.exp(-ce_loss)
        at      = self.alpha.gather(0, targets)
        loss    = at * (1 - pt) ** self.gamma * ce_loss
        return loss.mean() if self.reduction == 'mean' else loss


criterion  = FocalLoss()

In [13]:
# ----- Weighted CE --------------------------------------------------
# Calculá la proporción real de clases en el CSV que vayas a usar
# (si entrenás con un set balanceado, pesos = 1:1 y esta línea es opcional)
import numpy as np
labels_arr = np.array(train_dataset.labels)           # ya existe
neg, pos = (labels_arr == 0).sum(), (labels_arr == 1).sum()

w_spoof = neg / (neg + pos)   # peso inverso a la frecuencia
w_live  = pos / (neg + pos)

class_weights = torch.tensor([w_spoof, w_live], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f"Usando Weighted CE — pesos: spoof={w_spoof:.3f}, live={w_live:.3f}")
# --------------------------------------------------------------------


Usando Weighted CE — pesos: spoof=0.502, live=0.498


In [ ]:
# ----- Pairwise AUC Loss (pAUC) -------------------------------------
class PairwiseAUCLoss(nn.Module):
    """
    Surrogate differentiable loss que minimiza 1 - AUC.
    Basada en: Yuan et al., "Optimizing Area Under the ROC Curve", 2022.
    logits: salida del modelo (shape [B, 2]) antes de softmax
    labels: 0 = spoof, 1 = live
    """
    def __init__(self, partial=False, fpr=0.10, margin=0.0):
        super().__init__()
        self.partial = partial
        self.fpr = fpr
        self.margin = margin

    def forward(self, logits, labels):
        # Usamos prob_live = softmax(logits)[:,1]
        prob_live = torch.softmax(logits, dim=1)[:, 1]
        y = labels.float()

        pos_mask = y == 1
        neg_mask = y == 0
        pos_scores = prob_live[pos_mask]
        neg_scores = prob_live[neg_mask]

        if pos_scores.numel() == 0 or neg_scores.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        # Todas las combinaciones pos-neg  (|P| × |N|)
        diff = neg_scores.unsqueeze(0) - pos_scores.unsqueeze(1)  # P x N
        # margen opcional (>0 penaliza más si la diferencia es pequeña)
        diff = diff + self.margin
        loss_mat = torch.sigmoid(diff)    # prob(neg > pos)

        if self.partial:
            # Ordenamos neg_scores; nos quedamos con el percentil que
            # cubre FPR <= self.fpr  (los "peores" negativos)
            k = max(1, int(self.fpr * neg_scores.numel()))
            topk_neg_scores, _ = torch.topk(neg_scores, k=k, largest=True)
            diff_p = topk_neg_scores.unsqueeze(0) - pos_scores.unsqueeze(1)
            loss_mat = torch.sigmoid(diff_p)

        return loss_mat.mean()

# Elegí: parcial (≤10 % FPR)  o AUC completo
criterion = PairwiseAUCLoss(partial=True, fpr=0.10, margin=0.0)
print("Usando Pairwise pAUC Loss — región FPR ≤ 10 %")
# --------------------------------------------------------------------


In [19]:
# continua del bloque anterior, pero lo pongo aca para tenerlo separado
# 🧪 Ejecución del entrenamiento (se asume que todo está definido)
trained_model = train_model(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler, device,
        num_epochs=10, patience=6
)


Epoch 1/10 — LR 0.000500


TrainLoss 0.3229  ValLoss 0.1938  ValAcc 0.9384  F1_spoof 0.938
✅  Nuevo mejor modelo guardado

Epoch 2/10 — LR 0.000497


TrainLoss 0.1866  ValLoss 0.1605  ValAcc 0.9444  F1_spoof 0.943
✅  Nuevo mejor modelo guardado

Epoch 3/10 — LR 0.000488


TrainLoss 0.1518  ValLoss 0.1480  ValAcc 0.9468  F1_spoof 0.946
✅  Nuevo mejor modelo guardado

Epoch 4/10 — LR 0.000473


TrainLoss 0.1390  ValLoss 0.1319  ValAcc 0.9540  F1_spoof 0.954
✅  Nuevo mejor modelo guardado

Epoch 5/10 — LR 0.000452


...

TrainLoss 0.1291  ValLoss 0.1332  ValAcc 0.9516  F1_spoof 0.951
⏳  Sin mejora (2/6)

Epoch 7/10 — LR 0.000397


TrainLoss 0.1170  ValLoss 0.1195  ValAcc 0.9586  F1_spoof 0.958
✅  Nuevo mejor modelo guardado

Epoch 8/10 — LR 0.000363


TrainLoss 0.1147  ValLoss 0.1306  ValAcc 0.9520  F1_spoof 0.951
⏳  Sin mejora (1/6)

Epoch 9/10 — LR 0.000327


TrainLoss 0.1112  ValLoss 0.1252  ValAcc 0.9558  F1_spoof 0.955
⏳  Sin mejora (2/6)

Epoch 10/10 — LR 0.000289


TrainLoss 0.1083  ValLoss 0.1203  ValAcc 0.9580  F1_spoof 0.957
⏳  Sin mejora (3/6)


In [20]:
# Para 10k muestras, usando Weighted CE
# demoro 30 min aprox
# TrainLoss 0.1170  ValLoss 0.1195  ValAcc 0.9586  F1_spoof 0.958

In [24]:
import os, json, time
import numpy as np
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
from torch import amp


# 1) Creamos la arquitectura con la misma config de entrenamiento
best_model = models.mobilenet_v2(weights=None)           # no necesitamos pesos ImageNet
num_ftrs   = best_model.classifier[1].in_features
best_model.classifier[1] = nn.Linear(num_ftrs, 2)

# 2) Cargamos SOLO la parte de pesos
ckpt_path = os.path.join(RUN_DIR, 'best_model.pth')
ckpt = torch.load(ckpt_path, map_location=device)
best_model.load_state_dict(ckpt["model_state"])          # <- clave correcta

best_model.to(device).eval()
print("✅ Pesos cargados desde checkpoint")

def pad_metrics_from_cm(cm):
    """
    Etiquetas: 0 = spoof (ataque), 1 = live (bona fide)
    cm = [[TN, FP],
          [FN, TP]]
    APCER = cm[0,1] / (cm[0,0] + cm[0,1])
    BPCER = cm[1,0] / (cm[1,0] + cm[1,1])
    ACER  = (APCER + BPCER) / 2
    """
    attack_total = cm[0,0] + cm[0,1]
    bona_total   = cm[1,0] + cm[1,1]
    apcer = cm[0,1] / attack_total if attack_total > 0 else 0.0
    bpcer = cm[1,0] / bona_total   if bona_total > 0 else 0.0
    acer  = 0.5 * (apcer + bpcer)
    return apcer, bpcer, acer

def _metrics_from_preds(y_true, y_pred):
    cm = metrics.confusion_matrix(y_true, y_pred, labels=[0,1])
    apcer, bpcer, acer = pad_metrics_from_cm(cm)
    bal_acc = metrics.balanced_accuracy_score(y_true, y_pred)
    report  = metrics.classification_report(
        y_true, y_pred,
        target_names=["spoof", "live"],
        output_dict=True, zero_division=0
    )
    acc = (y_pred == y_true).mean()
    return {
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "apcer": float(apcer),
        "bpcer": float(bpcer),
        "acer": float(acer),
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }

@torch.inference_mode()
def test_model(model, test_loader, criterion, device, output_dir,
               threshold: float | None = None,
               target_apcer: float | None = 0.10):
    """
    threshold:
      - None → usa baseline 'argmax'
      - float en [0,1] → decide live si prob_live >= threshold
    target_apcer:
      - Si no es None, calcula un punto de operación que cumpla APCER <= target_apcer
    """
    os.makedirs(output_dir, exist_ok=True)
    model.eval()
    all_labels, all_probs = [], []
    all_preds_argmax = []
    test_loss = 0.0
    total_samples = 0

    use_cuda_amp = (device.type == 'cuda')

    start_time = time.time()
    num_batches = len(test_loader)

    for batch_idx, (inputs, labels) in tqdm(enumerate(test_loader),
                                            total=num_batches,
                                            desc="Testing", leave=True):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with amp.autocast(device_type='cuda', enabled=use_cuda_amp):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        probs_live = torch.softmax(outputs, dim=1)[:, 1]  # prob de clase "live"
        preds_argmax = outputs.argmax(dim=1)

        bs = inputs.size(0)
        test_loss += loss.item() * bs
        total_samples += bs

        all_probs.extend(probs_live.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_preds_argmax.extend(preds_argmax.cpu().numpy())

    total_time = time.time() - start_time
    avg_loss = test_loss / max(1, total_samples)
    print(f"\n✅ Test terminado en {total_time:.1f}s - Loss promedio: {avg_loss:.4f}")

    # Numpy
    y_true  = np.asarray(all_labels, dtype=int)
    p_live  = np.asarray(all_probs, dtype=float)
    y_argmx = np.asarray(all_preds_argmax, dtype=int)

    # AUC siempre por probas
    roc_auc = float(metrics.roc_auc_score(y_true, p_live))

    # Puntos de operación
    ops = {}

    # 1) Baseline: argmax
    ops["argmax"] = _metrics_from_preds(y_true, y_argmx)
    ops["argmax"]["threshold"] = None

    # 2) Youden (TPR-FPR máximo)
    fpr, tpr, thr = metrics.roc_curve(y_true, p_live, pos_label=1)
    youden_idx = int(np.argmax(tpr - fpr))
    thr_youden = float(thr[youden_idx])
    y_youden = (p_live >= thr_youden).astype(int)
    ops["youden"] = _metrics_from_preds(y_true, y_youden)
    ops["youden"]["threshold"] = thr_youden

    # 3) Min ACER en malla fina
    grid = np.linspace(0.0, 1.0, 1001)
    best = {"acer": 1.0, "thr": 0.5, "cm": None}
    for t in grid:
        y_hat = (p_live >= t).astype(int)
        cm = metrics.confusion_matrix(y_true, y_hat, labels=[0,1])
        apcer, bpcer, acer = pad_metrics_from_cm(cm)
        if acer < best["acer"]:
            best.update({"acer": acer, "thr": float(t), "cm": cm})
    y_acer = (p_live >= best["thr"]).astype(int)
    ops["acer_min"] = _metrics_from_preds(y_true, y_acer)
    ops["acer_min"]["threshold"] = best["thr"]

    # 4) APCER objetivo (si se pide)
    if target_apcer is not None:
        thr_target = None
        tuple_target = None
        for t in grid:
            y_hat = (p_live >= t).astype(int)
            cm = metrics.confusion_matrix(y_true, y_hat, labels=[0,1])
            apcer, bpcer, acer = pad_metrics_from_cm(cm)
            if apcer <= target_apcer:
                thr_target = float(t)
                tuple_target = (apcer, bpcer, acer, y_hat)
                break
        if thr_target is not None:
            y_t = tuple_target[3]
            ops[f"apcer<={target_apcer:.2f}"] = _metrics_from_preds(y_true, y_t)
            ops[f"apcer<={target_apcer:.2f}"]["threshold"] = thr_target
        else:
            ops[f"apcer<={target_apcer:.2f}"] = {
                "threshold": None,
                "note": f"No se alcanzó APCER <= {target_apcer:.2f} con thresholds uniformes"
            }

    # Selección para retorno (si threshold explícito, úsalo)
    if threshold is not None:
        thr_used = float(threshold)
        y_sel = (p_live >= thr_used).astype(int)
        cm_sel = metrics.confusion_matrix(y_true, y_sel, labels=[0,1])
        apcer, bpcer, acer = pad_metrics_from_cm(cm_sel)
        sel_metrics = _metrics_from_preds(y_true, y_sel)
        sel_metrics["threshold"] = thr_used
        sel_key = "custom_threshold"
        ops[sel_key] = sel_metrics
        cm_to_return = cm_sel
        report_for_return = sel_metrics["classification_report"]
    else:
        # por compat: devolvemos el baseline argmax
        cm_to_return = np.array(ops["argmax"]["confusion_matrix"])
        report_for_return = ops["argmax"]["classification_report"]

    # Guardado principal
    out_json = {
        "avg_loss": avg_loss,
        "roc_auc": roc_auc,
        "total_time_sec": total_time,
        "num_samples": int(total_samples),
        "num_batches": int(num_batches),
        "operating_points": ops
    }
    with open(os.path.join(output_dir, 'metrics_test.json'), 'w') as f:
        json.dump(out_json, f, indent=2)
    print("📄 Métricas de test (todas las variantes) guardadas en metrics_test.json")

    # Guardar predicciones crudas
    import pandas as pd
    df_preds = pd.DataFrame({
        "label": y_true,
        "pred_argmax": y_argmx,
        "prob_live": p_live
    })
    df_preds.to_csv(os.path.join(output_dir, "predictions_test.csv"), index=False)
    print("💾 Predicciones guardadas en predictions_test.csv")

    # Matriz de confusión del punto elegido (para la figura)
    plt.figure(figsize=(6,5))
    sns.heatmap(np.array(cm_to_return), annot=True, fmt='d', cmap='Blues',
                xticklabels=["spoof", "live"], yticklabels=["spoof", "live"])
    plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title("Matriz de Confusión")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'confusion_matrix.png'))
    plt.close()
    print("🖼️ Matriz de confusión guardada como imagen.")

    # Curva ROC (independiente del threshold)
    fpr, tpr, _ = metrics.roc_curve(y_true, p_live, pos_label=1)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Curva ROC (live=positivo)")
    plt.legend(loc="lower right"); plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'roc_curve.png'))
    plt.close()
    print("📈 Curva ROC guardada como imagen.")

    # Devolvemos: loss promedio, diccionario completo, y CM del punto elegido
    return avg_loss, out_json, np.array(cm_to_return)


✅ Pesos cargados desde checkpoint


In [25]:
# # Continuacion del bloque anterior. Testeo del modelo en otro dataset
# test_data_dir2 = '/content/drive/Othercomputers/My Mac/Data/test'


# 📦 Dataset de test desde CSV
test_dataset = CelebASpoofDatasetFromCSV(
    csv_path='/content/drive/Othercomputers/My Mac/Data/train/test_image_list.csv',
    transform=data_transforms['test'],
    max_samples=1000,
    balanced=False,
    use_bb=True
)

# 🔄 DataLoaders
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=6,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    drop_last=True
)



📄 CSV cargado: /content/drive/Othercomputers/My Mac/Data/train/test_image_list.csv
📊 Total imágenes cargadas: 1000


In [26]:
test_loss, test_report, test_cm = test_model(best_model, test_loader, criterion, device, output_dir=RUN_DIR)


Testing: 100%|██████████| 31/31 [03:49<00:00,  7.42s/it]



✅ Test terminado en 230.0s - Loss promedio: 0.1820
📄 Métricas de test (todas las variantes) guardadas en metrics_test.json
💾 Predicciones guardadas en predictions_test.csv
🖼️ Matriz de confusión guardada como imagen.
📈 Curva ROC guardada como imagen.


In [28]:
# # Continuacion del bloque anterior. Testeo del modelo en otro dataset
# test_data_dir2 = '/content/drive/Othercomputers/My Mac/Data/test'


# 📦 Dataset de test desde CSV
test_dataset = CelebASpoofDatasetFromCSV(
    csv_path='/content/drive/Othercomputers/My Mac/Data/train/test2_image_list_with_spoof_type.csv',
    transform=data_transforms['test'],
    max_samples=1300,
    balanced=False,
    use_bb=True
)

# 🔄 DataLoaders
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=6,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    drop_last=True
)



📄 CSV cargado: /content/drive/Othercomputers/My Mac/Data/train/test2_image_list_with_spoof_type.csv
📊 Total imágenes cargadas: 1300


In [29]:
test_loss, test_report, test_cm = test_model(best_model, test_loader, criterion, device, output_dir=RUN_DIR)


Testing: 100%|██████████| 40/40 [05:11<00:00,  7.78s/it]



✅ Test terminado en 311.4s - Loss promedio: 0.6789
📄 Métricas de test (todas las variantes) guardadas en metrics_test.json
💾 Predicciones guardadas en predictions_test.csv
🖼️ Matriz de confusión guardada como imagen.
📈 Curva ROC guardada como imagen.
